In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.9 MB/s eta 0:00:00


In [ ]:
import torch
import torchmetrics
import torch.nn as nn
import pandas as pd
import numpy as np
import torch.nn.functional as F

In [ ]:
!wget https://raw.githubusercontent.com/gazizovaa/egov-news-nlp/refs/heads/main/data/egov_news.csv

--2026-05-14 06:50:12--  https://raw.githubusercontent.com/gazizovaa/egov-news-nlp/refs/heads/main/data/egov_news.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2543848 (2.4M) [text/plain]
Saving to: ‘egov_news.csv’

egov_news.csv       100%[===================>]   2.43M  --.-KB/s    in 0.06s   

2026-05-14 06:50:12 (37.9 MB/s) - ‘egov_news.csv’ saved [2543848/2543848]



In [ ]:
df = pd.read_csv('egov_news.csv')
df

,Unnamed: 0,title,url,content,published date,views
0,0,Azərbaycan Qazaxıstanda keçirilən “Digital Bri...,https://www.e-gov.az/az/news/read/891,İnnovasiya və Rəqəmsal İnkişaf Agentliyinin nü...,16.10.2023,12922
1,1,İnnovasiya və Rəqəmsal İnkişaf Agentliyinin rə...,https://www.e-gov.az/az/news/read/893,Sentyabrın 17-19-da Bakıda Qlobal CIO Forumunu...,06.10.2023,12688
2,2,Azərbaycan dünyanın aparıcı tədbiri “TechCrunc...,https://www.e-gov.az/az/news/read/892,İnnovasiya və Rəqəmsal İnkişaf Agentliyi beynə...,02.10.2023,11768
3,3,Azərbaycan startapları Türkiyədə keçirilən “Te...,https://www.e-gov.az/az/news/read/888,İnnovasiya və Rəqəmsal İnkişaf Agentliyinin də...,05.09.2023,14643
4,4,İsrailin bir sıra aparıcı şirkət nümayəndələri...,https://www.e-gov.az/az/news/read/890,İnnovasiya və Rəqəmsal İnkişaf Agentliyinin tə...,02.09.2023,14307
...,...,...,...,...,...,...
823,823,“Elektron hökumət” portalında dövlət orqanları...,https://www.e-gov.az/az/news/read/7,Azərbaycan Respublikasının Prezidenti yanında ...,03.02.2014,3829
824,824,Artıq “Elektron hökumət” portalı üzərindən POS...,https://www.e-gov.az/az/news/read/10,“POS-terminalın onlayn qeydiyyatı” elektron xi...,28.01.2014,3797
825,825,“Elektron hökumət” portalı ilə bağlı növbəti ...,https://www.e-gov.az/az/news/read/11,01.01.2013-cü ildən 01.01.2014-cü ilə qədər “E...,15.01.2014,5244
826,826,2013-cü ildə “Elektron hökumət” portalına 262 ...,https://www.e-gov.az/az/news/read/16,Azərbaycan Respublikası Prezidenti İlham Əliye...,13.01.2014,2198


In [ ]:
df.isna().sum()

,0
Unnamed: 0,0
title,0
url,0
content,8
published date,0
views,0


In [ ]:
df['content'] = df['content'].fillna('')

In [ ]:
combined_texts = (df['title'] + " " + df['content']).tolist()

In [ ]:
combined_texts[:10]

['Azərbaycan Qazaxıstanda keçirilən “Digital Bridge” tədbirində təmsil olunub İnnovasiya və Rəqəmsal İnkişaf Agentliyinin nümayəndə heyəti 12-13 oktyabr tarixlərində “Digital Bridge” texnologiya tədbirində iştirak edib.  20 mindən çox iştirakçı, 300-dən çox İKT şirkəti və 100-dən çox investorun qatıldığı “Digital Bridge”də İRİA-nın nümayəndələri müxtəlif ölkələrin və şirkətlərin təmsilçiləri ilə tərəfdaşlıq və əməkdaşlıq imkanlarını dəyərləndiriblər.  Agentliyin bir qrup nümayəndə heyəti “GOVSTACK: RəqəmsalTransformasiyanın Sürətləndirilməsi” panel iclası, "Elektron hökuməttendensiyaları BMT-nin baxışı”, "Hökumətin gücləndirilməsi: Məlumata əsaslanan idarəetmə və milli süni intellekt" adlı dəyirmi masa müzakirələrində iştirak edib.  Müzakirələrdə elektron hökumətin əsas tendensiyalarını və ən yaxşı təcrübələri vurğulanıb, istifadəçilərin cəlb edilməsi və xidmətlərin qiymətləndirilməsi üçünverilənlərə əsaslanan, süni intellektlə dəstəklənən sistemlərdən istifadənin vacibliyi qeyd olunub

In [ ]:
full_text = "\n".join(combined_texts)

In [ ]:
print(f"Total text lengths: {len(full_text)}")

Total text lengths: 2139895


In [ ]:
full_text[:200]

'Azərbaycan Qazaxıstanda keçirilən “Digital Bridge” tədbirində təmsil olunub İnnovasiya və Rəqəmsal İnkişaf Agentliyinin nümayəndə heyəti 12-13 oktyabr tarixlərində “Digital Bridge” texnologiya tədbiri'

In [ ]:
vocab = sorted(set(full_text.lower()))
vocab[:50]

['\n',
 ' ',
 '!',
 '"',
 '#',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '=',
 '?',
 '@',
 '\\',
 '_',
 '`',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p']

In [ ]:
print(f"The number of distinct symbols: {len(vocab)}")

The number of distinct symbols: 119


In [ ]:
print("View first 50 symbols:", "".join(vocab[:50]) )

View first 50 symbols: 
 !"#%&'()*+,-./0123456789:;=?@\_`abcdefghijklmnop


In [ ]:
# stoi -> string to index
# nümunə -> {'!': 2}
stoi = {char: idx for idx, char in enumerate(vocab)}
stoi

{'\n': 0,
 ' ': 1,
 '!': 2,
 '"': 3,
 '#': 4,
 '%': 5,
 '&': 6,
 "'": 7,
 '(': 8,
 ')': 9,
 '*': 10,
 '+': 11,
 ',': 12,
 '-': 13,
 '.': 14,
 '/': 15,
 '0': 16,
 '1': 17,
 '2': 18,
 '3': 19,
 '4': 20,
 '5': 21,
 '6': 22,
 '7': 23,
 '8': 24,
 '9': 25,
 ':': 26,
 ';': 27,
 '=': 28,
 '?': 29,
 '@': 30,
 '\\': 31,
 '_': 32,
 '`': 33,
 'a': 34,
 'b': 35,
 'c': 36,
 'd': 37,
 'e': 38,
 'f': 39,
 'g': 40,
 'h': 41,
 'i': 42,
 'j': 43,
 'k': 44,
 'l': 45,
 'm': 46,
 'n': 47,
 'o': 48,
 'p': 49,
 'q': 50,
 'r': 51,
 's': 52,
 't': 53,
 'u': 54,
 'v': 55,
 'w': 56,
 'x': 57,
 'y': 58,
 'z': 59,
 '{': 60,
 '}': 61,
 '~': 62,
 '\xa0': 63,
 '«': 64,
 '\xad': 65,
 '·': 66,
 '¸': 67,
 '»': 68,
 'ç': 69,
 'é': 70,
 'ö': 71,
 'ü': 72,
 'ğ': 73,
 'ı': 74,
 'ş': 75,
 'ə': 76,
 'ʺ': 77,
 '̆': 78,
 '̇': 79,
 '̈': 80,
 '̧': 81,
 'а': 82,
 'в': 83,
 'д': 84,
 'е': 85,
 'з': 86,
 'и': 87,
 'к': 88,
 'л': 89,
 'н': 90,
 'о': 91,
 'п': 92,
 'р': 93,
 'с': 94,
 'т': 95,
 'у': 96,
 'х': 97,
 'ц': 98,
 'ш': 99,
 '

In [ ]:
# itos -> index to string
itos = {idx: char for idx, char in enumerate(vocab)}
itos

{0: '\n',
 1: ' ',
 2: '!',
 3: '"',
 4: '#',
 5: '%',
 6: '&',
 7: "'",
 8: '(',
 9: ')',
 10: '*',
 11: '+',
 12: ',',
 13: '-',
 14: '.',
 15: '/',
 16: '0',
 17: '1',
 18: '2',
 19: '3',
 20: '4',
 21: '5',
 22: '6',
 23: '7',
 24: '8',
 25: '9',
 26: ':',
 27: ';',
 28: '=',
 29: '?',
 30: '@',
 31: '\\',
 32: '_',
 33: '`',
 34: 'a',
 35: 'b',
 36: 'c',
 37: 'd',
 38: 'e',
 39: 'f',
 40: 'g',
 41: 'h',
 42: 'i',
 43: 'j',
 44: 'k',
 45: 'l',
 46: 'm',
 47: 'n',
 48: 'o',
 49: 'p',
 50: 'q',
 51: 'r',
 52: 's',
 53: 't',
 54: 'u',
 55: 'v',
 56: 'w',
 57: 'x',
 58: 'y',
 59: 'z',
 60: '{',
 61: '}',
 62: '~',
 63: '\xa0',
 64: '«',
 65: '\xad',
 66: '·',
 67: '¸',
 68: '»',
 69: 'ç',
 70: 'é',
 71: 'ö',
 72: 'ü',
 73: 'ğ',
 74: 'ı',
 75: 'ş',
 76: 'ə',
 77: 'ʺ',
 78: '̆',
 79: '̇',
 80: '̈',
 81: '̧',
 82: 'а',
 83: 'в',
 84: 'д',
 85: 'е',
 86: 'з',
 87: 'и',
 88: 'к',
 89: 'л',
 90: 'н',
 91: 'о',
 92: 'п',
 93: 'р',
 94: 'с',
 95: 'т',
 96: 'у',
 97: 'х',
 98: 'ц',
 99: 'ш',
 1

In [ ]:
print(stoi['ə'])

76


In [ ]:
print(stoi.get('<>', 'elə bir simvol yoxdur!'))

elə bir simvol yoxdur!


In [ ]:
# stoi-ları tensorlara çevirmə
def encode_text(text):
  return torch.tensor([stoi[char] for char in text.lower()])

In [ ]:
phrase = "Gülnarə Əzizova"
print(encode_text(text=phrase))

tensor([40, 72, 45, 47, 34, 51, 76,  1, 76, 59, 42, 59, 48, 55, 34])


In [ ]:
def decode_text(char_ids):
  return "".join([itos[char_id.item()] for char_id in char_ids])

In [ ]:
print(decode_text(encode_text(text=phrase)))

gülnarə əzizova


In [ ]:
from torch.utils.data import DataLoader, Dataset

In [ ]:
class CharDataset(Dataset):
  def __init__(self, text, window_length):
    self.encoded_text = encode_text(text)
    self.window_length = window_length

  def __len__(self):
    return len(self.encoded_text) - self.window_length

  def __getitem__(self, idx):
    if idx >= len(self):
      raise IndexError("dataset is out of range.")
    end = self.window_length + idx
    window = self.encoded_text[idx:end] # giriş
    target = self.encoded_text[idx + 1:end + 1] # çıxış
    return window, target

In [ ]:
window_length = 100
batch_size = 256

In [ ]:
n = len(full_text)
n

2139895

In [ ]:
train_text = full_text[:int(n * 0.80)]
valid_text = full_text[int(n * 0.80):int(n * 0.90)]
test_text = full_text[int(n * 0.90):]

In [ ]:
len(train_text), len(valid_text), len(test_text)

(1711916, 213989, 213990)

In [ ]:
train_set = CharDataset(train_text, window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

In [ ]:
valid_set = CharDataset(valid_text, window_length)
valid_loader = DataLoader(valid_set, batch_size=batch_size)

In [ ]:
test_set = CharDataset(test_text, window_length)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [ ]:
len(train_loader), len(valid_loader)

(6700, 837)

In [ ]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [ ]:
device

'cuda'

In [ ]:
# batch_first=True -> (batch, sequence, feature)
# self.output(outputs) -> (batch_size, sequence_length, vocab_size)
# self.output(outputs).permute(0, 2, 1) -> (batch_size, vocab_size, sequence_length)

In [ ]:
class EgovNewsModel(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim=16, hidden_dim=256, dropout=0.2):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim) # embedding layer
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout) # GRU
    self.output = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X):
    embeddings = self.embed(X)
    outputs, _states = self.gru(embeddings)
    return self.output(outputs).permute(0, 2, 1)

In [ ]:
torch.manual_seed(42)
model = EgovNewsModel(len(vocab)).to(device)

In [ ]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [ ]:
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index+1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch+1}/{n_epochs}                        "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train acc: {history['train_metrics'][-1]:.2%}, "
              f"valid acc: {history['valid_metrics'][-1]:.2%}")
    return history

In [ ]:
n_epochs = 5
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)

In [ ]:
history = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs)

Epoch 1/5                        train loss: 0.7431, train acc: 77.45%, valid acc: 72.83%
Epoch 2/5                        train loss: 0.7259, train acc: 77.92%, valid acc: 72.87%
Epoch 3/5                        train loss: 0.7172, train acc: 78.15%, valid acc: 72.94%
Epoch 4/5                        train loss: 0.7113, train acc: 78.30%, valid acc: 73.08%
Epoch 5/5                        train loss: 0.7067, train acc: 78.43%, valid acc: 73.04%


In [ ]:
torch.save(model.state_dict(), "egov_news_model.pt")

In [ ]:
# text generation
def next_char(model, text, temperature=1.0):
  # Use the last 'window_length' characters as context for generation
  context = text if len(text) < window_length else text[-window_length:]
  encoded = encode_text(context).unsqueeze(dim=0).to(device)
  with torch.no_grad():
    Y_logits = model(encoded)
    # Get logits for the next character, apply softmax along the correct dimension
    Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=0)
    predicted = torch.multinomial(Y_probas, num_samples=1).item()
  return itos[predicted]

In [ ]:
def extend_text(model, text, n_chars=200, temperature=1.0):
  model.eval()
  for _ in range(n_chars):
    text += next_char(model, text, temperature)
  return text

In [ ]:
# Test
seed_text = "elektron hökümət xidmətləri"

In [ ]:
extend_text(model, seed_text, n_chars=200, temperature=0.4)

'elektron hökümət xidmətləri və elektron hökumət sahəsində maraqlı olduğunu və bu sahədə daha çox istifadə olunan elektron xidmət və sosial i̇nnovasiyalar üzrə dövlət agentliyinin elektron hökumətin i̇nkişafı mərkəzi tərəfindən '

In [ ]:
# Sentiment Analysis
from sklearn.model_selection import train_test_split
import tokenizers

df_sent = pd.read_csv('egov_news.csv').dropna(subset=['content', 'views'])
median_views = df_sent['views'].median()
df_sent['label'] = (df_sent['views'] > median_views).astype(int)

In [ ]:
print(f"Median views: {median_views}")
print(f"Popularity: {df_sent['label'].sum()},  Less released: {(df_sent['label']==0).sum()}")
df_sent[['title', 'views', 'label']].head()

Median views: 3980.0
Popularity: 410,  Less released: 410


,title,views,label
0,Azərbaycan Qazaxıstanda keçirilən “Digital Bri...,12922,1
1,İnnovasiya və Rəqəmsal İnkişaf Agentliyinin rə...,12688,1
2,Azərbaycan dünyanın aparıcı tədbiri “TechCrunc...,11768,1
3,Azərbaycan startapları Türkiyədə keçirilən “Te...,14643,1
4,İsrailin bir sıra aparıcı şirkət nümayəndələri...,14307,1


In [ ]:
train_df, test_df = train_test_split(df_sent, test_size=0.2, random_state=42, stratify=df_sent['label'])
train_df, valid_df = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['label'])

In [ ]:
len(train_df), len(valid_df), len(test_df)

(590, 66, 164)

In [ ]:
# BPE -> Byte-Fair Encoding (tokenization algorithm)

In [ ]:
bpe_model = tokenizers.models.BPE(unk_token='<unk>')
egov_tokenizer = tokenizers.Tokenizer(bpe_model)
egov_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=2000, special_tokens=['<pad>', '<unk>']
)
egov_tokenizer.train_from_iterator(train_df['content'].str.lower().tolist(), bpe_trainer)
egov_tokenizer.enable_padding(pad_id=0, pad_token='<pad>')
egov_tokenizer.enable_truncation(max_length=500)

In [ ]:
print(f"Vocab size: {egov_tokenizer.get_vocab_size()}")
sample = egov_tokenizer.encode('elektron hökumət xidmətləri')
print('Tokens:', sample.tokens)

Vocab size: 2000
Tokens: ['elektron', 'hökumət', 'xidmətləri']


In [ ]:
class EgovSentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts  = dataframe['content'].str.lower().tolist()
        self.labels = dataframe['label'].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def egov_collate_fn(batch):
    texts, labels = zip(*batch)
    encodings = egov_tokenizer.encode_batch(list(texts))
    input_ids = torch.tensor([enc.ids for enc in encodings])
    attn_mask = torch.tensor([enc.attention_mask for enc in encodings])
    labels = torch.tensor(list(labels), dtype=torch.float32).unsqueeze(1) # (batch_size, 1)
    return {'input_ids': input_ids, 'attention_mask': attn_mask}, labels

In [ ]:
sent_batch_size   = 32
train_sent_loader = DataLoader(EgovSentimentDataset(train_df), batch_size=sent_batch_size, collate_fn=egov_collate_fn, shuffle=True)
valid_sent_loader = DataLoader(EgovSentimentDataset(valid_df), batch_size=sent_batch_size, collate_fn=egov_collate_fn)
test_sent_loader  = DataLoader(EgovSentimentDataset(test_df),  batch_size=sent_batch_size, collate_fn=egov_collate_fn)

In [ ]:
def train_sentiment(model, optimizer, train_loader, valid_loader, n_epochs=10):
    loss_fn  = nn.BCEWithLogitsLoss()
    accuracy = torchmetrics.Accuracy(task='binary').to(device)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        accuracy.reset()
        for idx, (encoding, labels) in enumerate(train_loader):
            encoding = {k: v.to(device) for k, v in encoding.items()}
            labels   = labels.to(device)
            y_pred   = model(encoding)
            loss     = loss_fn(y_pred, labels)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            accuracy.update(y_pred.sigmoid(), labels.int())
            print(f"\rBatch {idx+1}/{len(train_loader)}, "
                  f"loss={total_loss/(idx+1):.4f}", end='')
        train_acc = accuracy.compute().item()

        model.eval()
        accuracy.reset()
        with torch.no_grad():
            for encoding, labels in valid_loader:
                encoding = {k: v.to(device) for k, v in encoding.items()}
                labels   = labels.to(device)
                accuracy.update(model(encoding).sigmoid(), labels.int())
        valid_acc = accuracy.compute().item()

        print(f"\rEpoch {epoch+1}/{n_epochs}  "
              f"train acc: {train_acc:.2%},  valid acc: {valid_acc:.2%}")

In [ ]:
torch.manual_seed(42)

# Define the sentiment analysis model
class EgovSentimentModel(nn.Module):
  def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_classes=1):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True) # (batch_size, seq_len, vocab_size)
    self.classifier = nn.Linear(hidden_dim, num_classes)

  def forward(self, encoding):
    input_ids = encoding['input_ids']
    embeddings = self.embedding(input_ids)
    outputs, last_hidden_state = self.gru(embeddings)
    return self.classifier(last_hidden_state.squeeze(0))

egov_sent_model = EgovSentimentModel(egov_tokenizer.get_vocab_size()).to(device)
optimizer = torch.optim.NAdam(egov_sent_model.parameters(), lr=1e-3)
train_sentiment(egov_sent_model, optimizer, train_sent_loader, valid_sent_loader)

Epoch 1/10  train acc: 52.03%,  valid acc: 51.52%
Epoch 2/10  train acc: 58.81%,  valid acc: 45.45%
Epoch 3/10  train acc: 64.75%,  valid acc: 54.55%
Epoch 4/10  train acc: 62.54%,  valid acc: 51.52%
Epoch 5/10  train acc: 65.59%,  valid acc: 39.39%
Epoch 6/10  train acc: 65.42%,  valid acc: 50.00%
Epoch 7/10  train acc: 66.78%,  valid acc: 40.91%
Epoch 8/10  train acc: 69.66%,  valid acc: 45.45%
Epoch 9/10  train acc: 66.44%,  valid acc: 53.03%
Epoch 10/10  train acc: 70.68%,  valid acc: 51.52%
